#### ENRICH `SELLERS` FROM BRONZE TO SILVER

In [0]:
sellers_df = spark.table('ORDERS_SALES.BRONZE.SELLERS')
sellers_df.limit(1).display()

In [0]:
from pyspark.sql.functions import *

sellers_enriched_df = sellers_df.select(
    "seller_id",
    "seller_zip_code_prefix",
    upper(trim(col("seller_city"))).alias("seller_city"),
    upper(trim(col("seller_state"))).alias("seller_state"),
    "file_name",
    "file_path",
    current_timestamp().alias('load_timestamp')
).filter('seller_id IS NOT NULL')

display(sellers_enriched_df.limit(1))

In [0]:
sellers_enriched_df.createOrReplaceTempView('enriched_sellers_temp_vw')

#### INCREMENTAL LOAD `ORDERS_SALES.SILVER.SELLERS_ENRICHED`

In [0]:
%sql
MERGE INTO ORDERS_SALES.SILVER.SELLERS_ENRICHED tgt
USING enriched_sellers_temp_vw src
ON (src.seller_id = tgt.seller_id)
WHEN MATCHED
    THEN UPDATE SET
    tgt.seller_id = src.seller_id,
    tgt.seller_zip_code_prefix = src.seller_zip_code_prefix,
    tgt.seller_city = src.seller_city,
    tgt.seller_state = src.seller_state
WHEN NOT MATCHED THEN
INSERT (seller_id, seller_zip_code_prefix, seller_city, seller_state, origin_file_path, load_timestamp)
VALUES (src.seller_id, src.seller_zip_code_prefix, src.seller_city, src.seller_state, src.file_path, current_timestamp)

In [0]:
dbutils.notebook.exit('SUCCESSFULLY LOADED INTO ORDERS_SALES.SILVER.SELLERS_ENRICHED')